# GENERATE MONTHLY REPORT

### Section 1: Setup and Imports

In [120]:
# =============================================================================
# SECTION 1: SETUP AND IMPORTS
# =============================================================================
# This section imports all required libraries for:
# - Document creation (python-docx)
# - Data processing (pandas, numpy)
# - File operations (os, datetime)
# - Statistical analysis (scipy)

import pandas as pd
import numpy as np
from docx import Document
from docx.shared import Inches, Pt, Cm, RGBColor
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.enum.table import WD_TABLE_ALIGNMENT
from docx.enum.style import WD_STYLE_TYPE
from docx.oxml.ns import qn, nsdecls
from docx.oxml import parse_xml, OxmlElement
import os
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Import scipy for trend analysis
try:
    from scipy import stats
except ImportError:
    import subprocess
    subprocess.check_call(['pip', 'install', 'scipy', '--break-system-packages', '-q'])
    from scipy import stats

print("✓ Libraries imported successfully")

✓ Libraries imported successfully


### Section 2: Configuration and Path Setup

In [121]:
# =============================================================================
# SECTION 2: CONFIGURATION AND PATH SETUP
# =============================================================================
# Define all paths and configuration settings for Monthly Business Review
# Key differences from WBR:
# - Uses 6-month average instead of 6-week
# - Uses MoM (Month-over-Month) instead of WoW (Week-over-Week)
# - Trend periods: 6, 12, 24 months instead of weeks

# Base path configuration - assumes notebook is in 'notebooks' folder
BASE_PATH = os.path.dirname(os.getcwd())

# Folder paths
OUTPUTS_PATH = os.path.join(BASE_PATH, 'outputs')
VISUALIZATIONS_PATH = os.path.join(BASE_PATH, 'visualizations')
REPORTS_PATH = os.path.join(BASE_PATH, 'reports')
REF_DATE_FILE = os.path.join(BASE_PATH, 'analysis_ref_date.csv')
TARGET_FILE = os.path.join(BASE_PATH, 'target_reference.csv')  # Goal targets file

# Create reports folder if it doesn't exist
os.makedirs(REPORTS_PATH, exist_ok=True)

# Configuration settings for MBR
CONFIG = {
    # Analysis periods (in months for MBR)
    'months_for_average': 6,           # 6-month rolling average
    'near_threshold_pct': 0.05,        # ±5% = "Near" the average (configurable)
    
    # Spike/Dip detection thresholds
    'spike_threshold_pct': 0.25,       # ±25% vs 6-month average (configurable)
    'spike_threshold_std': 2,          # ±2 standard deviations
    
    # Chart display settings
    'chart_width_inches': 4.0,         # Chart width in document
    'charts_per_page': 2,              # Target charts per page
    
    # Trend analysis periods (in months)
    'trend_short_term': 6,             # 6-month trend
    'trend_medium_term': 12,           # 12-month trend
    'trend_long_term': 24,             # 24-month trend
    
    # Minimum data points required for trend analysis
    'min_points_for_trend': 4,
}

# Metrics with INVERTED color logic (lower is better)
# For these metrics: going DOWN = favorable (green), going UP = unfavorable (red)
INVERTED_METRICS = [
    'Active Unsubs', 'Auto Unsubs', 'Unsubscribes', 'Unsubscribed',
    'Leak Rate', 'List Unsub Rate', 'Blended Unsub Rate', 'Unsubscribe Rate',
    'Bounce Rate', 'Deal Unsubscribe Rate', 'Off-Market Unsubscribe Rate',
    'Podcast Unsubscribe Rate', 'Case Study Unsubscribe Rate',
    'Avg Lead Time', 'Cancellation Rate'
]

# Metrics that should use absolute values (stored as negative in CSV)
ABSOLUTE_VALUE_METRICS = [
    'Unsubscribed', 'Unsubscribes'
]

print(f"✓ Configuration loaded")
print(f"  Chart width: {CONFIG['chart_width_inches']}\"")
print(f"  Inverted metrics: {len(INVERTED_METRICS)}")
print(f"  Trend periods: {CONFIG['trend_short_term']}, {CONFIG['trend_medium_term']}, {CONFIG['trend_long_term']} months")
print(f"  Target file: {os.path.basename(TARGET_FILE)}")

✓ Configuration loaded
  Chart width: 4.0"
  Inverted metrics: 15
  Trend periods: 6, 12, 24 months
  Target file: target_reference.csv


### Section 3: Chart and Data Source Mapping

In [122]:
# =============================================================================
# SECTION 3: CHART AND DATA SOURCE MAPPING
# =============================================================================
# This mapping connects each chart to:
# - 'image': path to PNG file (relative to visualizations folder)
# - 'csv': path to data CSV file (relative to outputs folder)
# - 'metric': column name in CSV containing the metric values
# - 'date_col': column(s) for date reference
# - 'goal_metric': metric name in target_reference.csv (None if no goal)
#
# NOTE: For MBR, we use monthly chart folders (e.g., 'newsletter_monthly')
# and monthly data files (e.g., 'monthly_newsletter.csv')
#
# Chart filename conventions:
# - Newsletter series unsubscribe charts use: *_unsubscribe.png (not *_unsubscribe_rate.png)
# - Blended unsubscribe uses: blended_unsubscribe.png

CHART_CONFIG = {
    # =========================================================================
    # NEWSLETTER METRICS - Landing Page Funnel (Pages 3-4)
    # =========================================================================
    'Newsletter Visits': {
        'image': 'newsletter_monthly/monthly_visits.png',
        'csv': 'newsletter/monthly_newsletter.csv',
        'metric': 'Visits',
        'date_col': 'Date Range',
        'goal_metric': 'Monthly Newsletter Visits'
    },
    'Newsletter Visit Duration': {
        'image': 'newsletter_monthly/monthly_avg_duration.png',
        'csv': 'newsletter/monthly_newsletter.csv',
        'metric': 'Avg Duration',
        'date_col': 'Date Range',
        'goal_metric': 'Monthly Newsletter Visit Duration'
    },
    'Newsletter Bounce Rate': {
        'image': 'newsletter_monthly/monthly_bounce_rate.png',
        'csv': 'newsletter/monthly_newsletter.csv',
        'metric': 'Bounce Rate',
        'date_col': 'Date Range',
        'goal_metric': 'Monthly Newsletter Bounce Rate'
    },
    'Newsletter CVR %': {
        'image': 'general_newsletter_monthly/monthly_newsletter_cvr.png',
        'csv': 'general_newsletter/monthly_table.csv',
        'metric': 'Newsletter CVR %',
        'date_col': 'Date Range',
        'goal_metric': 'Monthly Newsletter CVR %'
    },
    
    # =========================================================================
    # NEWSLETTER METRICS - Engagement (Pages 5-6)
    # =========================================================================
    'Blended Open Rate': {
        'image': 'newsletter_series_monthly/blended_open_rate.png',
        'csv': 'newsletter_series/08_blended_monthly.csv',
        'metric': 'Open Rate',
        'date_col': 'Date Range',
        'goal_metric': 'Blended Open Rate'
    },
    'Blended Verified Click-Through Rate': {
        'image': 'newsletter_series_monthly/blended_verified_ctr.png',
        'csv': 'newsletter_series/08_blended_monthly.csv',
        'metric': 'Verified Click-Through Rate',
        'date_col': 'Date Range',
        'goal_metric': 'Blended Verified Click-Through Rate'
    },
    'Blended Unsub Rate': {
        'image': 'newsletter_series_monthly/blended_unsubscribe.png',
        'csv': 'newsletter_series/08_blended_monthly.csv',
        'metric': 'Unsubscribe Rate',
        'date_col': 'Date Range',
        'goal_metric': 'Blended Unsubscribe Rate'
    },
    
    # =========================================================================
    # NEWSLETTER METRICS - Growth & Churn (Pages 7-8)
    # =========================================================================
    'Growth Rate': {
        'image': 'general_newsletter_monthly/monthly_growth_rate.png',
        'csv': 'general_newsletter/monthly_table.csv',
        'metric': 'Growth Rate',
        'date_col': 'Date Range',
        'goal_metric': 'Monthly Growth Rate'
    },
    'New Subscribers': {
        'image': 'general_newsletter_monthly/monthly_subscribed.png',
        'csv': 'general_newsletter/monthly_table.csv',
        'metric': 'Subscribed',
        'date_col': 'Date Range',
        'goal_metric': 'Monthly Subscribed'
    },
    'Unsubscribes': {
        'image': 'general_newsletter_monthly/monthly_unsubscribed.png',
        'csv': 'general_newsletter/monthly_table.csv',
        'metric': 'Unsubscribed',
        'date_col': 'Date Range',
        'goal_metric': 'Monthly Unsubscribed'
    },
    'Leak Rate': {
        'image': 'general_newsletter_monthly/monthly_leak_rate.png',
        'csv': 'general_newsletter/monthly_table.csv',
        'metric': 'Leak Rate',
        'date_col': 'Date Range',
        'goal_metric': 'Monthly Leak Rate'
    },
    
    # =========================================================================
    # NEWSLETTER SERIES - New Deals (Pages 9-10)
    # =========================================================================
    'Deal Open Rate': {
        'image': 'newsletter_series_monthly/deal_open_rate.png',
        'csv': 'newsletter_series/02_deal_emails.csv',
        'metric': 'Open Rate',
        'date_col': 'Date',
        'goal_metric': 'Deal Open Rate'
    },
    'Deal Verified CTR': {
        'image': 'newsletter_series_monthly/deal_verified_ctr.png',
        'csv': 'newsletter_series/02_deal_emails.csv',
        'metric': 'Verified Click-Through Rate',
        'date_col': 'Date',
        'goal_metric': 'Deal Verified Click-Through Rate'
    },
    'Deal Unsubscribe Rate': {
        'image': 'newsletter_series_monthly/deal_unsubscribe.png',
        'csv': 'newsletter_series/02_deal_emails.csv',
        'metric': 'Unsubscribe Rate',
        'date_col': 'Date',
        'goal_metric': 'Deal Unsubscribe Rate'
    },
    
    # =========================================================================
    # NEWSLETTER SERIES - Off-Market (Pages 11-12)
    # =========================================================================
    'Off-Market Open Rate': {
        'image': 'newsletter_series_monthly/offmarket_open_rate.png',
        'csv': 'newsletter_series/03_offmarket_emails.csv',
        'metric': 'Open Rate',
        'date_col': 'Date',
        'goal_metric': 'Off-Market Open Rate'
    },
    'Off-Market Verified CTR': {
        'image': 'newsletter_series_monthly/offmarket_verified_ctr.png',
        'csv': 'newsletter_series/03_offmarket_emails.csv',
        'metric': 'Verified Click-Through Rate',
        'date_col': 'Date',
        'goal_metric': 'Off-Market Verified Click-Through Rate'
    },
    'Off-Market Unsubscribe Rate': {
        'image': 'newsletter_series_monthly/offmarket_unsubscribe.png',
        'csv': 'newsletter_series/03_offmarket_emails.csv',
        'metric': 'Unsubscribe Rate',
        'date_col': 'Date',
        'goal_metric': 'Off-Market Unsubscribe Rate'
    },
    
    # =========================================================================
    # NEWSLETTER SERIES - Podcasts (Pages 13-14)
    # =========================================================================
    'Podcast Open Rate': {
        'image': 'newsletter_series_monthly/podcast_open_rate.png',
        'csv': 'newsletter_series/04_podcast_emails.csv',
        'metric': 'Open Rate',
        'date_col': 'Date',
        'goal_metric': 'Podcast Open Rate'
    },
    'Podcast Verified CTR': {
        'image': 'newsletter_series_monthly/podcast_verified_ctr.png',
        'csv': 'newsletter_series/04_podcast_emails.csv',
        'metric': 'Verified Click-Through Rate',
        'date_col': 'Date',
        'goal_metric': 'Podcast Verified Click-Through Rate'
    },
    'Podcast Unsubscribe Rate': {
        'image': 'newsletter_series_monthly/podcast_unsubscribe.png',
        'csv': 'newsletter_series/04_podcast_emails.csv',
        'metric': 'Unsubscribe Rate',
        'date_col': 'Date',
        'goal_metric': 'Podcast Unsubscribe Rate'
    },
    
    # =========================================================================
    # NEWSLETTER SERIES - Case Study (Pages 15-16)
    # =========================================================================
    'Case Study Open Rate': {
        'image': 'newsletter_series_monthly/case_study_open_rate.png',
        'csv': 'newsletter_series/01_case_study_emails.csv',
        'metric': 'Open Rate',
        'date_col': 'Date',
        'goal_metric': 'Case Study Open Rate'
    },
    'Case Study Verified CTR': {
        'image': 'newsletter_series_monthly/case_study_verified_ctr.png',
        'csv': 'newsletter_series/01_case_study_emails.csv',
        'metric': 'Verified Click-Through Rate',
        'date_col': 'Date',
        'goal_metric': 'Case Study Verified Click-Through Rate'
    },
    'Case Study Unsubscribe Rate': {
        'image': 'newsletter_series_monthly/case_study_unsubscribe.png',
        'csv': 'newsletter_series/01_case_study_emails.csv',
        'metric': 'Unsubscribe Rate',
        'date_col': 'Date',
        'goal_metric': 'Case Study Unsubscribe Rate'
    },
    
    # =========================================================================
    # SALES METRICS - Lead Time (Page 17)
    # Chart path corrected: discovery_intro_blended_monthly
    # =========================================================================
    'Average Lead Time': {
        'image': 'discovery_intro_blended_monthly/monthly_avg_lead_time_completed.png',
        'csv': 'discovery_intro/monthly_discovery_intro_blended.csv',
        'metric': 'Avg Lead Time (Days)(Completed)',
        'date_col': 'Date Range',
        'goal_metric': 'Monthly Avg Lead Time (Days)(Completed)'
    },
    
    # =========================================================================
    # SALES METRICS - Deal Upgrade (Page 18)
    # =========================================================================
    'Deal Upgrade Visits': {
        'image': 'deal_upgrade_monthly/monthly_visits.png',
        'csv': 'deal_upgrade/monthly_deal_upgrade.csv',
        'metric': 'Visits',
        'date_col': 'Date Range',
        'goal_metric': 'Monthly DU Visits'
    },
    'Deal Upgrade CVR': {
        'image': 'deal_upgrade_monthly/monthly_conversion_rate.png',
        'csv': 'deal_upgrade/monthly_deal_upgrade.csv',
        'metric': 'Conversion Rate',
        'date_col': 'Date Range',
        'goal_metric': 'Monthly DU Conversion Rate'
    },
    
    # =========================================================================
    # SALES METRICS - Pro Site (Page 19)
    # =========================================================================
    'Pro Site Visits': {
        'image': 'pro_site_monthly/monthly_visits.png',
        'csv': 'pro_site/monthly_pro_site.csv',
        'metric': 'Visits',
        'date_col': 'Date Range',
        'goal_metric': 'Monthly PS Visits'
    },
    'Pro Site CVR': {
        'image': 'pro_site_monthly/monthly_conversion_rate.png',
        'csv': 'pro_site/monthly_pro_site.csv',
        'metric': 'Conversion Rate',
        'date_col': 'Date Range',
        'goal_metric': 'Monthly PS Conversion Rate'
    },
    
    # =========================================================================
    # SALES METRICS - Booked Calls (Page 20)
    # =========================================================================
    'Booked Calls - Closers (Discovery Call)': {
        'image': 'discovery_call_monthly/monthly_booked_calls_completed.png',
        'csv': 'discovery_intro/monthly_discovery_call.csv',
        'metric': 'Booked Calls (Completed)',
        'date_col': 'Date Range',
        'goal_metric': 'Monthly Booked Calls (Completed) - Discovery'
    },
    'Booked Calls - Setters (Intro Call)': {
        'image': 'intro_call_monthly/monthly_booked_calls_completed.png',
        'csv': 'discovery_intro/monthly_intro_call.csv',
        'metric': 'Booked Calls (Completed)',
        'date_col': 'Date Range',
        'goal_metric': 'Monthly Booked Calls (Completed) - Intro'
    },
    
    # =========================================================================
    # SALES METRICS - Sales Pipeline: Calls (Pages 21-22)
    # =========================================================================
    'Scheduled Calls': {
        'image': 'sales_tracker_monthly/monthly_sched_calls.png',
        'csv': 'sales_tracker/monthly_sales_tracker.csv',
        'metric': 'Sched. calls',
        'date_col': 'Date Range',
        'goal_metric': 'Monthly Scheduled calls'
    },
    'Live Calls': {
        'image': 'sales_tracker_monthly/monthly_live_calls.png',
        'csv': 'sales_tracker/monthly_sales_tracker.csv',
        'metric': 'Live calls',
        'date_col': 'Date Range',
        'goal_metric': 'Monthly Live Calls'
    },
    'Show Rate': {
        'image': 'sales_tracker_monthly/monthly_show_pct.png',
        'csv': 'sales_tracker/monthly_sales_tracker.csv',
        'metric': 'Show %',
        'date_col': 'Date Range',
        'goal_metric': 'Monthly Show %'
    },
    
    # =========================================================================
    # SALES METRICS - Offers (Page 23)
    # =========================================================================
    'Offers': {
        'image': 'sales_tracker_monthly/monthly_offers.png',
        'csv': 'sales_tracker/monthly_sales_tracker.csv',
        'metric': 'Offers',
        'date_col': 'Date Range',
        'goal_metric': 'Monthly Offers'
    },
    'Offer Rate': {
        'image': 'sales_tracker_monthly/monthly_offer_pct.png',
        'csv': 'sales_tracker/monthly_sales_tracker.csv',
        'metric': 'Offer %',
        'date_col': 'Date Range',
        'goal_metric': 'Monthly Offer %'
    },
    
    # =========================================================================
    # SALES METRICS - Closes (Page 24)
    # =========================================================================
    'Closes': {
        'image': 'sales_tracker_monthly/monthly_close_1.png',
        'csv': 'sales_tracker/monthly_sales_tracker.csv',
        'metric': 'Close 1',
        'date_col': 'Date Range',
        'goal_metric': 'Monthly Closes'
    },
    'Offer to Close Rate': {
        'image': 'sales_tracker_monthly/monthly_offer_to_close_pct.png',
        'csv': 'sales_tracker/monthly_sales_tracker.csv',
        'metric': 'Offer to Close %',
        'date_col': 'Date Range',
        'goal_metric': 'Monthly Offer to Close %'
    },
}

print(f"✓ Chart configuration loaded: {len(CHART_CONFIG)} charts mapped")
print(f"  Charts with goals: {sum(1 for c in CHART_CONFIG.values() if c.get('goal_metric'))}")

✓ Chart configuration loaded: 37 charts mapped
  Charts with goals: 37


### Section 4: Utility Functions - Value Parsing and Formatting

In [123]:
# =============================================================================
# SECTION 4: UTILITY FUNCTIONS - VALUE PARSING AND FORMATTING
# =============================================================================
# Helper functions for parsing values, formatting output, and determining
# metric types (percentage vs absolute values)

def parse_reference_dates(ref_file):
    """
    Parse analysis_ref_date.csv to extract MONTH date range for MBR.
    
    How it works:
    - Reads the CSV line by line
    - Extracts Month Start Date and Month End Date
    - Returns formatted date strings for filename and display
    
    Returns:
        dict with keys: month_start, month_end, date_range_filename, date_range_display
    """
    if not os.path.exists(ref_file):
        print(f"  ⚠ Reference file not found: {ref_file}")
        return None
    
    try:
        ref_df = pd.read_csv(ref_file, header=None)
        ref_dict = dict(zip(ref_df[0], ref_df[1]))
        
        # Get Month Start Date and Month End Date for MBR
        month_start = ref_dict.get('Month Start Date', '')
        month_end = ref_dict.get('Month End Date', '')
        
        # Format for filename: MM-DD-YYYY
        start_parts = month_start.split('/')
        end_parts = month_end.split('/')
        
        if len(start_parts) == 3 and len(end_parts) == 3:
            date_range_filename = f"{start_parts[0]}-{start_parts[1]}-{start_parts[2]}_to_{end_parts[0]}-{end_parts[1]}-{end_parts[2]}"
            date_range_display = f"{month_start} to {month_end}"
        else:
            date_range_filename = "unknown_date_range"
            date_range_display = "Unknown Date Range"
        
        return {
            'month_start': month_start,
            'month_end': month_end,
            'date_range_filename': date_range_filename,
            'date_range_display': date_range_display
        }
    except Exception as e:
        print(f"  ⚠ Error parsing reference dates: {e}")
        return None


def parse_value(value):
    """
    Parse a value that might be percentage string, number, or other format.
    
    Examples:
    - "50%" -> 50.0 (keeps as percentage number)
    - "+12.35%" -> 12.35
    - "33,821" -> 33821.0
    - 100 -> 100.0
    
    Returns:
        float or None if unparseable
    """
    if pd.isna(value):
        return None
    
    if isinstance(value, (int, float)):
        return float(value)
    
    value_str = str(value).strip().lstrip('+')
    
    # Handle percentage - just remove the % sign, keep the number
    if value_str.endswith('%'):
        try:
            return float(value_str.rstrip('%'))
        except ValueError:
            return None
    
    # Handle regular number (remove commas)
    try:
        return float(value_str.replace(',', ''))
    except ValueError:
        return None


def format_value(value, is_percentage=False, decimals=2):
    """
    Format numeric value for display in report.
    
    Args:
        value: The numeric value
        is_percentage: If True, add % sign
        decimals: Decimal places to show
    """
    if value is None or pd.isna(value):
        return 'N/A'
    
    if is_percentage:
        return f"{value:.{decimals}f}%"
    else:
        if abs(value) >= 1000:
            return f"{value:,.{decimals}f}"
        return f"{value:.{decimals}f}"


def format_change(value, is_percentage=False):
    """
    Format a change value with + or - sign for MoM comparisons.
    """
    if value is None or pd.isna(value):
        return 'N/A'
    
    sign = '+' if value > 0 else ''
    if is_percentage:
        return f"{sign}{value:.2f}%"
    return f"{sign}{value:.2f}"


def is_percentage_metric(metric_name):
    """
    Determine if metric should be displayed as percentage.
    Checks for keywords: rate, cvr, %, pct, percentage
    """
    metric_lower = metric_name.lower()
    pct_keywords = ['rate', 'cvr', '%', 'pct', 'percentage']
    return any(kw in metric_lower for kw in pct_keywords)


print("✓ Value parsing and formatting functions loaded")

✓ Value parsing and formatting functions loaded


### Section 4: Utility Functions - Value Parsing and Formatting

In [124]:
# =============================================================================
# SECTION 4: UTILITY FUNCTIONS - VALUE PARSING AND FORMATTING
# =============================================================================
# Helper functions for parsing values, formatting output, and determining
# metric types (percentage vs absolute values)

def parse_reference_dates(ref_file):
    """
    Parse analysis_ref_date.csv to extract MONTH date range for MBR.
    
    How it works:
    - Reads the CSV line by line
    - Extracts Month Start Date and Month End Date
    - Returns formatted date strings for filename and display
    
    Returns:
        dict with keys: month_start, month_end, date_range_filename, date_range_display
    """
    if not os.path.exists(ref_file):
        print(f"  ⚠ Reference file not found: {ref_file}")
        return None
    
    try:
        ref_df = pd.read_csv(ref_file, header=None)
        ref_dict = dict(zip(ref_df[0], ref_df[1]))
        
        # Get Month Start Date and Month End Date for MBR
        month_start = ref_dict.get('Month Start Date', '')
        month_end = ref_dict.get('Month End Date', '')
        
        # Format for filename: MM-DD-YYYY
        start_parts = month_start.split('/')
        end_parts = month_end.split('/')
        
        if len(start_parts) == 3 and len(end_parts) == 3:
            date_range_filename = f"{start_parts[0]}-{start_parts[1]}-{start_parts[2]}_to_{end_parts[0]}-{end_parts[1]}-{end_parts[2]}"
            date_range_display = f"{month_start} to {month_end}"
        else:
            date_range_filename = "unknown_date_range"
            date_range_display = "Unknown Date Range"
        
        return {
            'month_start': month_start,
            'month_end': month_end,
            'date_range_filename': date_range_filename,
            'date_range_display': date_range_display
        }
    except Exception as e:
        print(f"  ⚠ Error parsing reference dates: {e}")
        return None


def parse_value(value):
    """
    Parse a value that might be percentage string, number, or other format.
    
    Examples:
    - "50%" -> 50.0 (keeps as percentage number)
    - "+12.35%" -> 12.35
    - "33,821" -> 33821.0
    - 100 -> 100.0
    
    Returns:
        float or None if unparseable
    """
    if pd.isna(value):
        return None
    
    if isinstance(value, (int, float)):
        return float(value)
    
    value_str = str(value).strip().lstrip('+')
    
    # Handle percentage - just remove the % sign, keep the number
    if value_str.endswith('%'):
        try:
            return float(value_str.rstrip('%'))
        except ValueError:
            return None
    
    # Handle regular number (remove commas)
    try:
        return float(value_str.replace(',', ''))
    except ValueError:
        return None


def format_value(value, is_percentage=False, decimals=2):
    """
    Format numeric value for display in report.
    
    Args:
        value: The numeric value
        is_percentage: If True, add % sign
        decimals: Decimal places to show
    """
    if value is None or pd.isna(value):
        return 'N/A'
    
    if is_percentage:
        return f"{value:.{decimals}f}%"
    else:
        if abs(value) >= 1000:
            return f"{value:,.{decimals}f}"
        return f"{value:.{decimals}f}"


def format_change(value, is_percentage=False):
    """
    Format a change value with + or - sign for MoM comparisons.
    """
    if value is None or pd.isna(value):
        return 'N/A'
    
    sign = '+' if value > 0 else ''
    if is_percentage:
        return f"{sign}{value:.2f}%"
    return f"{sign}{value:.2f}"


def is_percentage_metric(metric_name):
    """
    Determine if metric should be displayed as percentage.
    Checks for keywords: rate, cvr, %, pct, percentage
    """
    metric_lower = metric_name.lower()
    pct_keywords = ['rate', 'cvr', '%', 'pct', 'percentage']
    return any(kw in metric_lower for kw in pct_keywords)


print("✓ Value parsing and formatting functions loaded")

✓ Value parsing and formatting functions loaded


### Section 5: Utility Functions - Trend and Spike Detection

In [125]:
# =============================================================================
# SECTION 5: UTILITY FUNCTIONS - TREND AND SPIKE DETECTION
# =============================================================================
# Statistical functions for analyzing monthly time series data
# Uses two methods for robust trend detection:
# 1. Linear Regression with p-value (for clean data)
# 2. First-half vs Second-half average comparison (for volatile data)

def calculate_trend(values, p_value_threshold=0.05, min_points=4, half_comparison_threshold=0.05):
    """
    Calculate trend direction using a HYBRID approach:
    
    Method 1: Linear Regression with p-value
    - If p-value ≤ threshold → use slope direction (statistically significant)
    
    Method 2: First-half vs Second-half average comparison (for volatile data)
    - If p-value > threshold (volatile), compare averages of first half vs second half
    - If second half avg is significantly higher → Rising
    - If second half avg is significantly lower → Falling
    - Otherwise → Stable
    
    Args:
        values: List of values (oldest to newest)
        p_value_threshold: Significance level for linear regression (default 0.05)
        min_points: Minimum data points required
        half_comparison_threshold: Minimum % difference between halves to detect trend (default 5%)
    
    Returns:
        str: 'rising', 'falling', 'stable', or 'insufficient data'
    """
    clean_values = [v for v in values if v is not None and not pd.isna(v)]
    
    if len(clean_values) < min_points:
        return 'insufficient data'
    
    n = len(clean_values)
    x = np.arange(n)
    y = np.array(clean_values)
    
    # Method 1: Linear Regression
    slope, intercept, r_value, p_value, std_err = stats.linregress(x, y)
    
    # If p-value is significant, use linear regression result
    if p_value <= p_value_threshold:
        if slope > 0:
            return 'rising'
        elif slope < 0:
            return 'falling'
        else:
            return 'stable'
    
    # Method 2: First-half vs Second-half comparison (for volatile data)
    # Split data into two halves
    mid_point = n // 2
    first_half = clean_values[:mid_point]
    second_half = clean_values[mid_point:]
    
    if len(first_half) < 2 or len(second_half) < 2:
        return 'stable'
    
    first_half_avg = np.mean(first_half)
    second_half_avg = np.mean(second_half)
    
    # Calculate percentage change between halves
    if first_half_avg != 0:
        pct_change = (second_half_avg - first_half_avg) / abs(first_half_avg)
    else:
        pct_change = 0
    
    # Classify based on percentage change between halves
    if pct_change > half_comparison_threshold:
        return 'rising'
    elif pct_change < -half_comparison_threshold:
        return 'falling'
    else:
        return 'stable'


def calculate_multi_period_trends(values):
    """
    Calculate trends over multiple time horizons using hybrid approach.
    
    For MBR (Monthly), we analyze:
    - 6-month trend (short term)
    - 12-month trend (medium term)  
    - 24-month trend (long term)
    
    Different thresholds for different periods:
    - 6mo: 8% change between halves needed (short term = more noise tolerance)
    - 12mo: 6% change between halves needed
    - 24mo: 5% change between halves needed (long term = smaller changes are meaningful)
    
    Returns:
        dict with 'short', 'medium', 'long' keys, each containing:
        - 'period': number of months
        - 'trend': 'rising', 'falling', 'stable', or 'insufficient data'
    """
    short_period = CONFIG['trend_short_term']    # 6 months
    medium_period = CONFIG['trend_medium_term']  # 12 months
    long_period = CONFIG['trend_long_term']      # 24 months
    
    trends = {
        'short': {'period': short_period, 'trend': 'insufficient data'},
        'medium': {'period': medium_period, 'trend': 'insufficient data'},
        'long': {'period': long_period, 'trend': 'insufficient data'},
    }
    
    num_points = len(values)
    
    # Short-term trend (need at least 6 points)
    if num_points >= short_period:
        trends['short']['trend'] = calculate_trend(
            values[-short_period:], 
            p_value_threshold=0.05, 
            min_points=4,
            half_comparison_threshold=0.08  # 8% for 6 months
        )
    
    # Medium-term trend (need at least 12 points)
    if num_points >= medium_period:
        trends['medium']['trend'] = calculate_trend(
            values[-medium_period:], 
            p_value_threshold=0.05, 
            min_points=6,
            half_comparison_threshold=0.06  # 6% for 12 months
        )
    
    # Long-term trend (need at least 24 points)
    if num_points >= long_period:
        trends['long']['trend'] = calculate_trend(
            values[-long_period:], 
            p_value_threshold=0.05, 
            min_points=12,
            half_comparison_threshold=0.05  # 5% for 24 months
        )
    
    return trends


def classify_vs_average(current, six_month_avg, near_threshold=None):
    """
    Classify current value relative to 6-month average.
    
    Args:
        current: Current month's value
        six_month_avg: 6-month rolling average
        near_threshold: Percentage threshold for "Near" classification (default from CONFIG)
    
    Returns:
        str: 'High CV vs 6Mo Avg', 'Low CV vs 6Mo Avg', or 'Normal'
        
    CV = Current Value
    """
    if near_threshold is None:
        near_threshold = CONFIG['near_threshold_pct']
        
    if current is None or six_month_avg is None or six_month_avg == 0:
        return 'Normal'
    
    pct_diff = (current - six_month_avg) / abs(six_month_avg)
    
    # Above threshold (>5% by default) = High
    if pct_diff > near_threshold:
        return 'High CV vs 6Mo Avg'
    # Below threshold (<-5% by default) = Low  
    elif pct_diff < -near_threshold:
        return 'Low CV vs 6Mo Avg'
    # Within ±5% = Near/Normal
    return 'Normal (Near Avg)'


def detect_spike_or_dip(current, six_month_avg, six_month_std):
    """
    Detect if current month represents a spike or dip using two rules:
    
    1. Statistical rule: ±2 standard deviations from 6-month mean
    2. Relative rule: ±25% vs 6-month average (configurable)
    
    Args:
        current: Current month's value
        six_month_avg: 6-month rolling average
        six_month_std: 6-month standard deviation
    
    Returns:
        str: 'Spike', 'Dip', or 'No anomaly'
    """
    if current is None or six_month_avg is None:
        return 'No anomaly'
    
    spike_pct_threshold = CONFIG['spike_threshold_pct']  # 25%
    spike_std_threshold = CONFIG['spike_threshold_std']  # 2
    
    # Rule 1: Statistical (±2 std deviations)
    if six_month_std is not None and six_month_std > 0:
        upper_bound_std = six_month_avg + (spike_std_threshold * six_month_std)
        lower_bound_std = six_month_avg - (spike_std_threshold * six_month_std)
        
        if current > upper_bound_std:
            return 'Spike'
        elif current < lower_bound_std:
            return 'Dip'
    
    # Rule 2: Relative (±25% from average)
    if six_month_avg != 0:
        pct_diff = (current - six_month_avg) / abs(six_month_avg)
        
        if pct_diff > spike_pct_threshold:
            return 'Spike'
        elif pct_diff < -spike_pct_threshold:
            return 'Dip'
    
    return 'No anomaly'


print("✓ Trend and spike detection functions loaded (hybrid: p-value + half comparison)")

✓ Trend and spike detection functions loaded (hybrid: p-value + half comparison)


### Section 6: Data Loading Functions

In [126]:
# =============================================================================
# SECTION 6: DATA LOADING FUNCTIONS
# =============================================================================
# Functions to load CSV and Excel files, extract time series data
# Data is filtered to only include records up to the reporting month end date

# Global variable for reporting month end date filter
REPORTING_MONTH_END = None

def set_reporting_month_end(date_info):
    """
    Set the global reporting month end date used to filter data.
    Only data up to this date will be included in analysis.
    """
    global REPORTING_MONTH_END
    month_end_str = date_info.get('month_end', '')
    
    formats = ['%m/%d/%Y', '%Y-%m-%d', '%m/%d/%y']
    for fmt in formats:
        try:
            REPORTING_MONTH_END = datetime.strptime(month_end_str, fmt)
            print(f"  Data filter: Only including data up to {REPORTING_MONTH_END.strftime('%m/%d/%Y')}")
            return
        except ValueError:
            continue
    
    print(f"  ⚠ Could not parse month end date: {month_end_str}")
    REPORTING_MONTH_END = None


def load_csv_data(csv_path):
    """Load and clean a CSV file, removing empty rows."""
    if not os.path.exists(csv_path):
        return None
    try:
        df = pd.read_csv(csv_path)
        df = df.dropna(how='all')
        return df
    except Exception as e:
        print(f"  ⚠ Error loading CSV: {e}")
        return None


def load_excel_metrics(xlsx_path):
    """Load metrics table from Excel file, removing empty rows and columns."""
    if not os.path.exists(xlsx_path):
        return None
    try:
        df = pd.read_excel(xlsx_path)
        df = df.dropna(how='all').dropna(axis=1, how='all')
        return df
    except Exception as e:
        print(f"  ⚠ Error loading Excel: {e}")
        return None


def parse_date_for_sorting(date_str):
    """
    Parse date string to datetime object for sorting and filtering.
    For date ranges like "11/01/2025 - 11/30/2025", extracts the END date.
    """
    if pd.isna(date_str):
        return None
    
    date_str = str(date_str).strip()
    
    # Extract END date from date ranges
    if ' - ' in date_str:
        date_str = date_str.split(' - ')[-1].strip()
    
    # Skip non-date strings
    if date_str.lower().startswith('month') or len(date_str) < 6:
        return None
    
    formats = ['%m/%d/%Y', '%Y-%m-%d', '%m/%d/%y', '%d/%m/%Y', '%m-%d-%Y']
    
    for fmt in formats:
        try:
            return datetime.strptime(date_str, fmt)
        except ValueError:
            continue
    
    return None


def extract_metric_series(df, metric_col, date_col):
    """
    Extract time series for a metric column from dataframe.
    
    Filtering:
    - Only includes data where date <= REPORTING_MONTH_END
    - Data beyond the reporting month is ignored
    
    Sorting:
    - Returns data sorted chronologically (oldest first, newest last)
    - values[-1] = most recent month (the reporting month)
    - values[-2] = previous month
    
    Returns:
        List of (date_string, value) tuples sorted oldest to newest
    """
    global REPORTING_MONTH_END
    
    if df is None or metric_col not in df.columns:
        return []
    
    raw_data = []
    
    # Handle two-column date format: [Start Date, End Date]
    if isinstance(date_col, list) and len(date_col) == 2:
        start_col, end_col = date_col
        if start_col in df.columns and end_col in df.columns:
            for idx, row in df.iterrows():
                date_display = f"{row[start_col]} - {row[end_col]}"
                value = parse_value(row[metric_col])
                sort_date = parse_date_for_sorting(str(row[end_col]))
                if value is not None:
                    raw_data.append({
                        'date_display': date_display,
                        'value': value,
                        'sort_date': sort_date
                    })
    # Handle single date column
    elif date_col in df.columns:
        for idx, row in df.iterrows():
            date_display = str(row[date_col])
            value = parse_value(row[metric_col])
            sort_date = parse_date_for_sorting(date_display)
            if value is not None:
                raw_data.append({
                    'date_display': date_display,
                    'value': value,
                    'sort_date': sort_date
                })
    
    if not raw_data:
        return []
    
    # Separate records with and without parseable dates
    with_dates = [d for d in raw_data if d['sort_date'] is not None]
    without_dates = [d for d in raw_data if d['sort_date'] is None]
    
    # Filter: only include data up to reporting month end date
    if REPORTING_MONTH_END is not None:
        with_dates = [d for d in with_dates if d['sort_date'] <= REPORTING_MONTH_END]
    
    # Sort chronologically: oldest first, newest last
    with_dates.sort(key=lambda x: x['sort_date'])
    
    # Build result list
    result = [(d['date_display'], d['value']) for d in with_dates]
    result.extend([(d['date_display'], d['value']) for d in without_dates])
    
    return result


print("✓ Data loading functions loaded")

✓ Data loading functions loaded


### Section 7: Chart Insight Generation

In [127]:
# =============================================================================
# SECTION 7: CHART INSIGHT GENERATION
# =============================================================================
# Functions to calculate insights from chart data with multi-period trends
# Handles decimal-to-percent conversion for specific files/metrics
# Customized for MBR with 6-month analysis window
# NOW INCLUDES: Goal attainment from target_reference.csv

# Define which CSV files and metrics store values as decimals (need *100)
# Only general_newsletter/monthly_table.csv stores these metrics as decimals
DECIMAL_METRICS = {
    'general_newsletter/monthly_table.csv': ['Growth Rate', 'Newsletter CVR %', 'Leak Rate', 'Unsub Rate']
}

# Global variable to store goal targets
GOAL_TARGETS = {}

def load_goal_targets():
    """
    Load goal targets from target_reference.csv.
    Parses the file and stores metric -> target value mapping.
    
    Returns dict: {metric_name: target_value}
    """
    global GOAL_TARGETS
    
    if not os.path.exists(TARGET_FILE):
        print(f"  ⚠ Target file not found: {TARGET_FILE}")
        return {}
    
    try:
        # Read the file line by line to handle the non-standard format
        targets = {}
        with open(TARGET_FILE, 'r') as f:
            lines = f.readlines()
        
        for line in lines:
            line = line.strip()
            # Skip empty lines, headers, and section dividers
            if not line or line.startswith('Metric') or line.startswith('=') or line.startswith('NEWSLETTER') or line.startswith('DEAL') or line.startswith('PRO') or line.startswith('DISCOVERY') or line.startswith('SALES') or line.startswith('GENERAL'):
                continue
            
            # Parse metric,value pairs
            if ',' in line:
                parts = line.split(',', 1)
                metric_name = parts[0].strip()
                if len(parts) > 1 and parts[1].strip():
                    try:
                        target_value = float(parts[1].strip())
                        targets[metric_name] = target_value
                    except ValueError:
                        # No target value for this metric
                        pass
        
        GOAL_TARGETS = targets
        return targets
    
    except Exception as e:
        print(f"  ⚠ Error loading target file: {e}")
        return {}


def get_goal_target(goal_metric_name):
    """
    Get the goal target value for a metric.
    
    Args:
        goal_metric_name: The metric name as it appears in target_reference.csv
        
    Returns:
        Target value (float) or None if no goal defined
    """
    global GOAL_TARGETS
    
    # Load targets if not already loaded
    if not GOAL_TARGETS:
        load_goal_targets()
    
    if not goal_metric_name:
        return None
    
    return GOAL_TARGETS.get(goal_metric_name)


def needs_decimal_conversion(csv_path, metric_name):
    """
    Check if this metric needs decimal-to-percent conversion.
    
    Only general_newsletter/monthly_table.csv stores certain metrics as decimals.
    All other CSV files are already in percentage form.
    """
    for csv_key, metrics in DECIMAL_METRICS.items():
        if csv_key in csv_path:
            if any(m.lower() == metric_name.lower() for m in metrics):
                return True
    return False


def get_chart_data_insights(chart_name, config):
    """
    Load chart data and calculate all insights for the reporting month.
    
    Handles:
    - Absolute value metrics (converts negative to positive)
    - Inverted metrics (where lower is better)
    - Decimal to percent conversion (only for specific files/metrics)
    - Multi-period trend analysis with Linear Regression
    - Spike/Dip detection
    - Goal attainment comparison
    
    Returns dict with:
    - current_value, previous_value
    - mom_change_abs, mom_change_pct
    - six_month_avg, six_month_std
    - vs_average classification
    - spike_dip detection
    - multi-period trends
    - goal_target, goal_met, goal_diff_pct (if goal defined)
    """
    csv_path = os.path.join(OUTPUTS_PATH, config['csv'])
    df = load_csv_data(csv_path)
    
    insights = {
        'chart_name': chart_name,
        'csv_source': config['csv'],
        'image_source': config['image'],
        'metric': config['metric'],
        'data_available': False,
        'data_points': 0
    }
    
    if df is None:
        insights['error'] = 'CSV file not found'
        return insights
    
    # Extract filtered and sorted time series
    series = extract_metric_series(df, config['metric'], config['date_col'])
    
    if len(series) < 2:
        insights['error'] = f'Not enough data points ({len(series)})'
        insights['data_points'] = len(series)
        return insights
    
    insights['data_available'] = True
    insights['data_points'] = len(series)
    
    # Extract values (series is now 2-tuple: (date, value))
    values = [v for _, v in series]
    
    # Check if this metric should use absolute values
    is_absolute_metric = any(abs_m.lower() in config['metric'].lower() 
                            for abs_m in ABSOLUTE_VALUE_METRICS)
    
    if is_absolute_metric:
        values = [abs(v) for v in values]
    
    # Current = last value (reporting month)
    # Previous = second-to-last value (month before)
    current = values[-1]
    previous = values[-2]
    
    # Check if this specific file/metric needs decimal conversion
    is_pct_metric = is_percentage_metric(config['metric'])
    needs_conversion = needs_decimal_conversion(config['csv'], config['metric'])
    
    if is_pct_metric and needs_conversion:
        # This metric is stored as decimal (e.g., 0.20 for 20%), convert to percentage
        current_display = current * 100
        previous_display = previous * 100
    else:
        # Already in percentage form or not a percentage metric
        current_display = current
        previous_display = previous
    
    insights['current_value'] = current_display
    insights['previous_value'] = previous_display
    insights['current_date'] = series[-1][0]
    insights['previous_date'] = series[-2][0]
    insights['is_absolute_metric'] = is_absolute_metric
    insights['is_pct_metric'] = is_pct_metric
    insights['needs_conversion'] = needs_conversion
    
    # Month-over-month change calculation
    if previous != 0:
        insights['mom_change_pct'] = (current - previous) / abs(previous)
        insights['mom_change_abs'] = current_display - previous_display
    else:
        insights['mom_change_pct'] = None
        insights['mom_change_abs'] = current_display - previous_display
    
    # 6-month statistics (convert to display values if needed)
    last_6 = values[-6:] if len(values) >= 6 else values
    if is_pct_metric and needs_conversion:
        last_6_display = [v * 100 for v in last_6]
    else:
        last_6_display = last_6
    insights['six_month_avg'] = np.mean(last_6_display)
    insights['six_month_std'] = np.std(last_6_display) if len(last_6_display) >= 2 else 0
    
    # Classification vs 6-month average
    insights['vs_average'] = classify_vs_average(current_display, insights['six_month_avg'])
    
    # Spike/Dip detection
    insights['spike_dip'] = detect_spike_or_dip(current_display, insights['six_month_avg'], insights['six_month_std'])
    
    # Multi-period trend analysis
    insights['trends'] = calculate_multi_period_trends(values)
    
    # ==========================================================================
    # GOAL ATTAINMENT - Compare current value to target from target_reference.csv
    # ==========================================================================
    goal_metric_name = config.get('goal_metric')
    if goal_metric_name:
        target = get_goal_target(goal_metric_name)
        if target is not None:
            insights['goal_target'] = target
            insights['goal_metric_name'] = goal_metric_name
            
            # Check if goal is met (for inverted metrics, lower is better)
            is_inverted = any(inv.lower() in config['metric'].lower() for inv in INVERTED_METRICS)
            
            if is_inverted:
                # For inverted metrics: actual <= target means goal met
                insights['goal_met'] = current_display <= target
            else:
                # For normal metrics: actual >= target means goal met
                insights['goal_met'] = current_display >= target
            
            # Calculate difference from goal
            if target != 0:
                insights['goal_diff_pct'] = ((current_display - target) / abs(target)) * 100
            else:
                insights['goal_diff_pct'] = 0 if current_display == 0 else 100
    
    return insights


def format_trend_with_arrow(trend):
    """Convert trend string to display format with arrow."""
    if trend == 'rising':
        return '↑ Rising'
    elif trend == 'falling':
        return '↓ Falling'
    elif trend == 'stable':
        return '→ Stable'
    else:
        return None  # Return None for insufficient data


print("✓ Chart insight generation functions loaded")
print(f"  Goal targets loaded: {len(GOAL_TARGETS) if GOAL_TARGETS else 'will load on first use'}")

✓ Chart insight generation functions loaded
  Goal targets loaded: will load on first use


### Section 8: Generate Chart Insight Text (MBR Format)

In [128]:
# =============================================================================
# SECTION 8: GENERATE CHART INSIGHT TEXT (MBR FORMAT)
# =============================================================================
# Generates human-readable insight paragraphs from calculated insights
# Customized for Monthly Business Review with all required elements:
# - Current vs Previous (MoM)
# - 6-month average comparison
# - Goal attainment (if goals defined)
# - Multi-period trends (6, 12, 24 months)

def generate_chart_insight_text(insights, metric_name):
    """
    Generate human-readable insight paragraph from calculated insights.
    
    MBR Output format (paragraph style):
    - Line 1: Current value vs previous month (MoM change)
    - Line 2: Performance vs 6-month average with classification
    - Line 3: Goal attainment (if goal defined, otherwise skipped)
    - Line 4: Trend direction over 6/12/24 months
    
    Features:
    - Zero change shows "no change from previous month"
    - Percentages always show 2 decimal places (XX.XX%)
    - Uses (favorable)/(unfavorable) labels for inverted metrics
    - Goal attainment shows met/missed with variance
    """
    if not insights.get('data_available', False):
        error_msg = insights.get('error', 'Data unavailable')
        data_points = insights.get('data_points', 0)
        if data_points > 0 and data_points < 6:
            return f"⚠ Not enough data points to assess trend ({data_points} available, need at least 6)"
        return f"⚠ {error_msg}"
    
    is_pct = insights.get('is_pct_metric', is_percentage_metric(metric_name))
    is_inverted = any(inv.lower() in metric_name.lower() for inv in INVERTED_METRICS)
    
    current = insights['current_value']
    previous = insights['previous_value']
    mom_pct = insights.get('mom_change_pct')
    mom_abs = insights.get('mom_change_abs', 0)
    six_avg = insights['six_month_avg']
    vs_avg = insights['vs_average']
    trends = insights.get('trends', {})
    data_points = insights['data_points']
    
    lines = []
    
    # Format values based on metric type (always 2 decimal places for percentages)
    if is_pct:
        curr_str = f"{current:.2f}%"
        prev_str = f"{previous:.2f}%"
        avg_str = f"{six_avg:.2f}%"
        change_str = f"{abs(mom_abs):.2f}%"
    else:
        curr_str = f"{current:,.0f}" if current >= 100 else f"{current:.2f}"
        prev_str = f"{previous:,.0f}" if abs(previous) >= 100 else f"{previous:.2f}"
        avg_str = f"{six_avg:,.0f}" if six_avg >= 100 else f"{six_avg:.2f}"
        change_str = f"{abs(mom_abs):,.0f}" if abs(mom_abs) >= 100 else f"{abs(mom_abs):.2f}"
    
    # ==========================================================================
    # LINE 1: Current month vs previous month (MoM change)
    # ==========================================================================
    if mom_pct is not None:
        # Check for zero or near-zero change
        if abs(mom_pct) < 0.001:  # Less than 0.1% change
            lines.append(f"• Current month: {curr_str} vs {prev_str} last month (no change MoM)")
        else:
            direction = "+" if mom_abs > 0 else ""
            pct_change_str = f"{mom_pct*100:+.1f}%"
            
            if is_inverted:
                # For inverted metrics: down = good, up = bad
                quality = "(favorable)" if mom_pct < 0 else "(unfavorable)"
            else:
                # For normal metrics: up = good, down = bad
                quality = "(favorable)" if mom_pct > 0 else "(unfavorable)"
            
            lines.append(f"• Current month: {curr_str} vs {prev_str} last month ({direction}{change_str}, {pct_change_str} MoM) {quality}")
    else:
        lines.append(f"• Current month: {curr_str} (previous: {prev_str})")
    
    # ==========================================================================
    # LINE 2: Performance vs 6-month average with classification
    # ==========================================================================
    if six_avg != 0:
        pct_from_avg = ((current - six_avg) / abs(six_avg)) * 100
        above_below = "above" if pct_from_avg > 0 else "below"
        lines.append(f"• Performance is {abs(pct_from_avg):.1f}% {above_below} the 6-month average (avg = {avg_str}) → {vs_avg}")
    else:
        lines.append(f"• 6-month average: {avg_str} → {vs_avg}")
    
    # ==========================================================================
    # LINE 3: Goal attainment (only if goal defined)
    # ==========================================================================
    if 'goal_target' in insights:
        target = insights['goal_target']
        goal_met = insights.get('goal_met', False)
        goal_diff_pct = insights.get('goal_diff_pct', 0)
        
        # Format target value
        if is_pct:
            target_str = f"{target:.2f}%"
        else:
            target_str = f"{target:,.0f}" if target >= 100 else f"{target:.2f}"
        
        if goal_met:
            if is_inverted:
                # For inverted metrics, being under target is good
                lines.append(f"• Goal met: Yes (target = {target_str}; actual = {curr_str}; {abs(goal_diff_pct):.1f}% below goal)")
            else:
                # For normal metrics, being above target is good
                lines.append(f"• Goal met: Yes (target = {target_str}; actual = {curr_str}; +{abs(goal_diff_pct):.1f}% above goal)")
        else:
            if is_inverted:
                # For inverted metrics, being over target is bad
                lines.append(f"• Goal missed: (target = {target_str}; actual = {curr_str}; +{abs(goal_diff_pct):.1f}% above goal)")
            else:
                # For normal metrics, being under target is bad
                lines.append(f"• Goal missed by {abs(goal_diff_pct):.1f}% (target = {target_str}; actual = {curr_str})")
    
    # ==========================================================================
    # LINE 4: Trend direction (6, 12, and 24 months)
    # ==========================================================================
    trend_parts = []
    
    # Short-term (6 months)
    short_trend = trends.get('short', {}).get('trend', 'insufficient data')
    short_formatted = format_trend_with_arrow(short_trend)
    if short_formatted:
        trend_parts.append(f"{short_formatted} (6mo)")
    elif data_points < 6:
        trend_parts.append(f"Not enough data for 6-month trend ({data_points} points)")
    
    # Medium-term (12 months)
    medium_trend = trends.get('medium', {}).get('trend', 'insufficient data')
    medium_formatted = format_trend_with_arrow(medium_trend)
    if medium_formatted:
        trend_parts.append(f"{medium_formatted} (12mo)")
    
    # Long-term (24 months)
    long_trend = trends.get('long', {}).get('trend', 'insufficient data')
    long_formatted = format_trend_with_arrow(long_trend)
    if long_formatted:
        trend_parts.append(f"{long_formatted} (24mo)")
    
    if trend_parts:
        lines.append(f"• Trend: {' | '.join(trend_parts)}")
    else:
        lines.append(f"• Trend: Insufficient data for trend analysis (only {data_points} data points)")
    
    return '\n'.join(lines)


print("✓ Chart insight text generation function loaded")

✓ Chart insight text generation function loaded


### Section 9: Section Summary Generation (Executive Summary)

In [129]:
# =============================================================================
# SECTION 9: SECTION SUMMARY GENERATION (EXECUTIVE SUMMARY)
# =============================================================================
# Functions to generate executive summary bullets from metrics tables
# Analyzes top improvements and concerns based on MoM changes

def generate_section_summary(df, section_name='metrics'):
    """
    Generate summary bullets from metrics table for executive summary.
    
    Analyzes the MoM column to find:
    - Top 3-5 improvements (largest positive changes)
    - Top 3-5 concerns (largest negative changes)
    
    Args:
        df: DataFrame with metrics (must have a MoM or similar column)
        section_name: 'newsletter' or 'sales' for labeling
    
    Returns:
        dict with 'improvements', 'concerns', 'summary' keys
    """
    if df is None or df.empty:
        return {
            'improvements': [],
            'concerns': [],
            'summary': f'No {section_name} data available'
        }
    
    # Find the MoM/change column
    change_col = None
    for col in df.columns:
        col_lower = str(col).lower()
        if 'mom' in col_lower or 'change' in col_lower:
            change_col = col
            break
    
    # If no change column found, use last column
    if change_col is None:
        change_col = df.columns[-1]
    
    # Find the metric name column (usually first column)
    metric_col = df.columns[0]
    
    # Parse changes and separate into improvements/concerns
    improvements = []
    concerns = []
    
    for _, row in df.iterrows():
        metric_name = str(row[metric_col])
        change_val = row[change_col]
        
        try:
            # Parse the change value
            change_str = str(change_val).replace('%', '').replace('+', '').strip()
            change_num = float(change_str)
            
            # Check if this is an inverted metric
            is_inverted = any(inv.lower() in metric_name.lower() for inv in INVERTED_METRICS)
            
            if is_inverted:
                # For inverted metrics: negative change = improvement
                if change_num < -1:  # More than 1% decrease
                    improvements.append((metric_name, change_num, change_val))
                elif change_num > 1:  # More than 1% increase
                    concerns.append((metric_name, change_num, change_val))
            else:
                # For normal metrics: positive change = improvement
                if change_num > 1:  # More than 1% increase
                    improvements.append((metric_name, change_num, change_val))
                elif change_num < -1:  # More than 1% decrease
                    concerns.append((metric_name, change_num, change_val))
        except (ValueError, TypeError):
            continue
    
    # Sort by magnitude
    improvements.sort(key=lambda x: abs(x[1]), reverse=True)
    concerns.sort(key=lambda x: abs(x[1]), reverse=True)
    
    # Take top 3
    top_improvements = improvements[:3]
    top_concerns = concerns[:3]
    
    # Build summary text
    summary_parts = []
    
    if top_improvements:
        summary_parts.append("Key Improvements:")
        for metric, _, change in top_improvements:
            summary_parts.append(f"  • {metric}: {change} MoM")
    
    if top_concerns:
        summary_parts.append("Areas of Concern:")
        for metric, _, change in top_concerns:
            summary_parts.append(f"  • {metric}: {change} MoM")
    
    return {
        'improvements': top_improvements,
        'concerns': top_concerns,
        'summary': '\n'.join(summary_parts) if summary_parts else 'No significant changes detected'
    }


def generate_executive_summary(newsletter_df, sales_df):
    """
    Generate executive summary combining newsletter and sales highlights.
    
    Returns:
        str: Combined summary text for both sections (6-10 bullets total)
    """
    newsletter_summary = generate_section_summary(newsletter_df, 'newsletter')
    sales_summary = generate_section_summary(sales_df, 'sales')
    
    summary_text = []
    summary_text.append("NEWSLETTER PERFORMANCE")
    summary_text.append(newsletter_summary['summary'])
    summary_text.append("")
    summary_text.append("SALES PERFORMANCE")
    summary_text.append(sales_summary['summary'])
    
    return '\n'.join(summary_text)


print("✓ Section summary generation functions loaded")

✓ Section summary generation functions loaded


### Section 10: Document Formatting Functions

In [130]:
# =============================================================================
# SECTION 10: DOCUMENT FORMATTING FUNCTIONS
# =============================================================================
# Functions for document layout, tables, and page management
# Includes color formatting for MoM columns
# Includes source file display below tables

def add_page_break(doc):
    """
    Add a page break to the document.
    Creates a new paragraph with a page break.
    Use insert_page_break_before() when possible to avoid blank pages.
    """
    doc.add_page_break()


def insert_page_break_before(paragraph):
    """
    Insert a page break at the START of an existing paragraph.
    This prevents blank pages by attaching the break to content.
    """
    run = paragraph.runs[0] if paragraph.runs else paragraph.add_run()
    br = OxmlElement('w:br')
    br.set(qn('w:type'), 'page')
    run._r.insert(0, br)


def set_cell_shading(cell, color_hex):
    """Apply background shading to a table cell."""
    shading = parse_xml(f'<w:shd {nsdecls("w")} w:fill="{color_hex}"/>')
    cell._tc.get_or_add_tcPr().append(shading)


def add_formatted_table(doc, df, title=None, color_mom=True, source_file=None):
    """
    Add a formatted data table to the document.
    Color codes the MoM column based on positive/negative values.
    
    Args:
        doc: Document object
        df: DataFrame with table data
        title: Optional title above table
        color_mom: Whether to color code the MoM column
        source_file: Optional source file path to display below table
    """
    if title:
        p = doc.add_paragraph()
        run = p.add_run(title)
        run.bold = True
        run.font.size = Pt(14)
        p.paragraph_format.space_after = Pt(6)
    
    table = doc.add_table(rows=1, cols=len(df.columns))
    table.style = 'Table Grid'
    
    # Header row
    header_cells = table.rows[0].cells
    for i, col_name in enumerate(df.columns):
        header_cells[i].text = str(col_name)
        header_cells[i].paragraphs[0].runs[0].bold = True
        header_cells[i].paragraphs[0].runs[0].font.size = Pt(11)
        set_cell_shading(header_cells[i], "D9E2F3")
    
    # Find the MoM column index (usually last or second-to-last)
    mom_col_idx = None
    for i, col in enumerate(df.columns):
        col_lower = str(col).lower()
        if 'mom' in col_lower or 'wow' in col_lower or col_lower in ['mom', 'mom/mtd']:
            mom_col_idx = i
            break
    
    # If not found by name, assume it's the last column with % values
    if mom_col_idx is None:
        mom_col_idx = len(df.columns) - 1
    
    # Data rows
    for _, row in df.iterrows():
        row_cells = table.add_row().cells
        for i, value in enumerate(row):
            cell = row_cells[i]
            cell.text = str(value) if pd.notna(value) else ""
            
            for paragraph in cell.paragraphs:
                for run in paragraph.runs:
                    run.font.size = Pt(11)
            
            # Color code MoM column
            if color_mom and i == mom_col_idx:
                try:
                    val_str = str(value).replace('%', '').replace('+', '').strip()
                    val_num = float(val_str)
                    metric_name = str(row.iloc[0]).lower() if len(row) > 0 else ""
                    
                    is_inverted = any(inv.lower() in metric_name for inv in INVERTED_METRICS)
                    
                    if is_inverted:
                        # For inverted metrics: negative = good (green), positive = bad (red)
                        if val_num < 0:
                            set_cell_shading(cell, "C6EFCE")  # Green
                        elif val_num > 0:
                            set_cell_shading(cell, "FFC7CE")  # Red
                    else:
                        # For normal metrics: positive = good (green), negative = bad (red)
                        if val_num > 0:
                            set_cell_shading(cell, "C6EFCE")  # Green
                        elif val_num < 0:
                            set_cell_shading(cell, "FFC7CE")  # Red
                except (ValueError, TypeError):
                    pass
    
    # Add source file below table (dark gray, small font)
    if source_file:
        p = doc.add_paragraph()
        run = p.add_run(f"Source: {source_file}")
        run.font.size = Pt(7)
        run.font.color.rgb = RGBColor(100, 100, 100)  # Dark gray
        p.paragraph_format.space_before = Pt(2)
        p.paragraph_format.space_after = Pt(6)


def add_page_numbers(doc):
    """Add page numbers to document footer."""
    for section in doc.sections:
        footer = section.footer
        footer.is_linked_to_previous = False
        
        p = footer.paragraphs[0] if footer.paragraphs else footer.add_paragraph()
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        
        run1 = p.add_run("Page ")
        run1.font.size = Pt(10)
        
        fldChar1 = OxmlElement('w:fldChar')
        fldChar1.set(qn('w:fldCharType'), 'begin')
        instrText1 = OxmlElement('w:instrText')
        instrText1.text = "PAGE"
        fldChar2 = OxmlElement('w:fldChar')
        fldChar2.set(qn('w:fldCharType'), 'separate')
        fldChar3 = OxmlElement('w:fldChar')
        fldChar3.set(qn('w:fldCharType'), 'end')
        
        run2 = p.add_run()
        run2._r.append(fldChar1)
        run2._r.append(instrText1)
        run2._r.append(fldChar2)
        run2._r.append(fldChar3)
        run2.font.size = Pt(10)
        
        run3 = p.add_run(" of ")
        run3.font.size = Pt(10)
        
        fldChar4 = OxmlElement('w:fldChar')
        fldChar4.set(qn('w:fldCharType'), 'begin')
        instrText2 = OxmlElement('w:instrText')
        instrText2.text = "NUMPAGES"
        fldChar5 = OxmlElement('w:fldChar')
        fldChar5.set(qn('w:fldCharType'), 'separate')
        fldChar6 = OxmlElement('w:fldChar')
        fldChar6.set(qn('w:fldCharType'), 'end')
        
        run4 = p.add_run()
        run4._r.append(fldChar4)
        run4._r.append(instrText2)
        run4._r.append(fldChar5)
        run4._r.append(fldChar6)
        run4.font.size = Pt(10)


print("✓ Document formatting functions loaded")

✓ Document formatting functions loaded


### Section 11: Chart with Insights Function

In [131]:
# =============================================================================
# SECTION 11: CHART WITH INSIGHTS FUNCTION
# =============================================================================
# Adds chart image and insights to the document
# Charts are sized to fit 2 per page with their insights
# Includes full source paths for both chart image and data CSV

def add_chart_with_insights(doc, chart_name, config, source_log, page_break_before=False):
    """
    Add a chart image with auto-generated insights to the document.
    Chart + insights are kept compact to fit 2 per page.
    
    Includes:
    - Chart source path (e.g., visualizations/newsletter_monthly/monthly_visits.png)
    - Data source path (e.g., outputs/newsletter/monthly_newsletter.csv)
    """
    # Chart title
    p = doc.add_paragraph()
    run = p.add_run(chart_name)
    run.bold = True
    run.font.size = Pt(11)
    p.paragraph_format.space_before = Pt(0)
    p.paragraph_format.space_after = Pt(2)
    
    # Add page break to this paragraph if requested
    if page_break_before:
        insert_page_break_before(p)
    
    # Chart image - smaller size
    image_path = os.path.join(VISUALIZATIONS_PATH, config['image'])
    
    if os.path.exists(image_path):
        pic = doc.add_picture(image_path, width=Inches(CONFIG['chart_width_inches']))
        # Reduce space after image
        last_para = doc.paragraphs[-1]
        last_para.paragraph_format.space_after = Pt(1)
        source_log.append(f"  Chart: {config['image']}")
    else:
        doc.add_paragraph(f"[Chart not found: {config['image']}]")
        source_log.append(f"  Chart: {config['image']} (NOT FOUND)")
    
    # Add full source paths below chart (dark gray, small font)
    # Chart source: visualizations/[folder]/[file].png
    # Data source: outputs/[folder]/[file].csv
    chart_source = f"visualizations/{config['image']}"
    data_source = f"outputs/{config['csv']}"
    
    p = doc.add_paragraph()
    # Chart source
    run = p.add_run(f"Chart: {chart_source}\n")
    run.font.size = Pt(7)
    run.font.color.rgb = RGBColor(100, 100, 100)  # Dark gray
    # Data source
    run = p.add_run(f"Source: {data_source}")
    run.font.size = Pt(7)
    run.font.color.rgb = RGBColor(100, 100, 100)  # Dark gray
    p.paragraph_format.space_before = Pt(0)
    p.paragraph_format.space_after = Pt(2)
    
    # Generate insights
    insights = get_chart_data_insights(chart_name, config)
    insight_text = generate_chart_insight_text(insights, config['metric'])
    
    # Insight label
    p = doc.add_paragraph()
    run = p.add_run("Insights:")
    run.bold = True
    run.font.size = Pt(9)
    p.paragraph_format.space_before = Pt(2)
    p.paragraph_format.space_after = Pt(1)
    
    # Insight text - compact
    p = doc.add_paragraph(insight_text)
    p.paragraph_format.left_indent = Inches(0.15)
    p.paragraph_format.space_after = Pt(8)
    for run in p.runs:
        run.font.size = Pt(9)
    
    source_log.append(f"  Data: {config['csv']} → {config['metric']}")
    
    return True


print("✓ Chart with insights function loaded")

✓ Chart with insights function loaded


### Section 12: Cover Page and Executive Summary

In [132]:
# =============================================================================
# SECTION 12: COVER PAGE AND EXECUTIVE SUMMARY
# =============================================================================
# Creates the report cover page with title, dates, and table of contents
# Creates executive summary on page 2

def add_table_of_contents(doc):
    """
    Add a Table of Contents field to the document.
    User must right-click and 'Update Field' in Word to populate.
    """
    paragraph = doc.add_paragraph()
    run = paragraph.add_run()
    
    fldChar1 = OxmlElement('w:fldChar')
    fldChar1.set(qn('w:fldCharType'), 'begin')
    
    instrText = OxmlElement('w:instrText')
    instrText.set(qn('xml:space'), 'preserve')
    instrText.text = ' TOC \\o "1-3" \\h \\z \\u '
    
    fldChar2 = OxmlElement('w:fldChar')
    fldChar2.set(qn('w:fldCharType'), 'separate')
    
    fldChar3 = OxmlElement('w:fldChar')
    fldChar3.set(qn('w:fldCharType'), 'end')
    
    run._r.append(fldChar1)
    run._r.append(instrText)
    run._r.append(fldChar2)
    
    placeholder_run = paragraph.add_run("Right-click and select 'Update Field' to generate Table of Contents")
    placeholder_run.italic = True
    placeholder_run.font.color.rgb = RGBColor(128, 128, 128)
    
    run2 = paragraph.add_run()
    run2._r.append(fldChar3)
    
    return doc


def create_cover_page(doc, date_info):
    """
    Create cover page with title, reporting month dates, and table of contents.
    """
    # Title - Monthly Business Review (MBR)
    title = doc.add_paragraph()
    title.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = title.add_run("Monthly Business Review (MBR)")
    run.bold = True
    run.font.size = Pt(28)
    
    doc.add_paragraph()
    
    # Reporting month
    subtitle = doc.add_paragraph()
    subtitle.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = subtitle.add_run(f"Reporting Month: {date_info['date_range_display']}")
    run.font.size = Pt(16)
    
    doc.add_paragraph()
    
    # Generation timestamp
    gen_date = doc.add_paragraph()
    gen_date.alignment = WD_ALIGN_PARAGRAPH.CENTER
    run = gen_date.add_run(f"Report Generated: {datetime.now().strftime('%B %d, %Y at %I:%M %p')}")
    run.font.size = Pt(11)
    run.italic = True
    
    doc.add_paragraph()
    doc.add_paragraph()
    
    # Table of Contents header
    toc_header = doc.add_paragraph()
    run = toc_header.add_run("Table of Contents")
    run.bold = True
    run.font.size = Pt(14)
    
    # Add TOC field
    add_table_of_contents(doc)
    
    return doc


def create_executive_summary(doc, newsletter_df, sales_df):
    """
    Create Executive Summary on page 2 with key improvements and concerns.
    """
    add_page_break(doc)
    
    doc.add_heading("Executive Summary", level=1)
    
    # Generate summary from metrics tables
    exec_summary = generate_executive_summary(newsletter_df, sales_df)
    
    for line in exec_summary.split('\n'):
        if line.strip():
            p = doc.add_paragraph(line)
            p.paragraph_format.space_after = Pt(3)
            # Bold section headers
            if line.isupper() or (line.endswith(':') and not line.startswith(' ')):
                for run in p.runs:
                    run.bold = True
    
    return doc


print("✓ Cover page and executive summary functions loaded")

✓ Cover page and executive summary functions loaded


### Section 13: Newsletter Section Generation

In [133]:
# =============================================================================
# SECTION 13: NEWSLETTER SECTION GENERATION
# =============================================================================
# Creates the Newsletter Metrics section with:
# - Landing Page Funnel (Pages 3-4)
# - Engagement (Pages 5-6)
# - Growth & Churn (Pages 7-8)
# - Newsletter Series: New Deals, Off-Market, Podcasts, Case Study (Pages 9-16)
#
# Section title uses Heading 1 style so it appears in Table of Contents

def add_section_title(doc, title_text, page_break_before=True):
    """
    Add a centered section title as Heading 1 (appears in Table of Contents).
    Used for main section headers (Section 1, Section 2).
    
    Creates a title that appears in the CENTER of the page.
    Uses Heading 1 style so it's included in the Table of Contents.
    """
    if page_break_before:
        doc.add_page_break()
    
    # Add multiple spacer paragraphs to push title to vertical center
    for _ in range(10):
        spacer = doc.add_paragraph()
        spacer.paragraph_format.space_before = Pt(0)
        spacer.paragraph_format.space_after = Pt(15)
    
    # Add the section title as Heading 1 (for TOC) - centered
    h = doc.add_heading(title_text, level=1)
    h.alignment = WD_ALIGN_PARAGRAPH.CENTER
    
    # Style the heading
    for run in h.runs:
        run.bold = True
        run.font.size = Pt(28)
        run.font.color.rgb = RGBColor(31, 73, 125)  # Dark blue color
    
    h.paragraph_format.space_before = Pt(0)
    h.paragraph_format.space_after = Pt(0)
    
    return h


def create_newsletter_section(doc, source_log):
    """Create Newsletter Metrics section with all charts and insights."""
    
    # Section 1 header - CENTERED, as Heading 1 (appears in TOC)
    add_section_title(doc, "Section 1: Newsletter Metrics", page_break_before=True)
    
    # =========================================================================
    # LANDING PAGE FUNNEL (Pages 3-4)
    # =========================================================================
    h = doc.add_heading("Landing Page Funnel", level=2)
    insert_page_break_before(h)
    source_log.append(f"\n--- Landing Page Funnel Charts ---")
    
    # Page 3: Newsletter Visits, Visit Duration
    add_chart_with_insights(doc, 'Newsletter Visits', CHART_CONFIG['Newsletter Visits'], source_log)
    add_chart_with_insights(doc, 'Newsletter Visit Duration', CHART_CONFIG['Newsletter Visit Duration'], source_log)
    
    # Page 4: Bounce Rate, CVR %
    add_chart_with_insights(doc, 'Newsletter Bounce Rate', CHART_CONFIG['Newsletter Bounce Rate'], source_log, page_break_before=True)
    add_chart_with_insights(doc, 'Newsletter CVR %', CHART_CONFIG['Newsletter CVR %'], source_log)
    
    # =========================================================================
    # ENGAGEMENT METRICS (Pages 5-6)
    # =========================================================================
    h = doc.add_heading("Engagement Metrics", level=2)
    insert_page_break_before(h)
    source_log.append(f"\n--- Engagement Charts ---")
    
    # Page 5: Blended Open Rate, Blended Verified CTR
    add_chart_with_insights(doc, 'Blended Open Rate', CHART_CONFIG['Blended Open Rate'], source_log)
    add_chart_with_insights(doc, 'Blended Verified Click-Through Rate', CHART_CONFIG['Blended Verified Click-Through Rate'], source_log)
    
    # Page 6: Blended Unsub Rate
    add_chart_with_insights(doc, 'Blended Unsub Rate', CHART_CONFIG['Blended Unsub Rate'], source_log, page_break_before=True)
    
    # =========================================================================
    # GROWTH & CHURN (Pages 7-8)
    # =========================================================================
    h = doc.add_heading("Growth & Churn", level=2)
    insert_page_break_before(h)
    source_log.append(f"\n--- Growth & Churn Charts ---")
    
    # Page 7: Growth Rate, New Subscribers
    add_chart_with_insights(doc, 'Growth Rate', CHART_CONFIG['Growth Rate'], source_log)
    add_chart_with_insights(doc, 'New Subscribers', CHART_CONFIG['New Subscribers'], source_log)
    
    # Page 8: Unsubscribes, Leak Rate
    add_chart_with_insights(doc, 'Unsubscribes', CHART_CONFIG['Unsubscribes'], source_log, page_break_before=True)
    add_chart_with_insights(doc, 'Leak Rate', CHART_CONFIG['Leak Rate'], source_log)
    
    # =========================================================================
    # NEW DEALS (Pages 9-10)
    # =========================================================================
    h = doc.add_heading("Newsletter Series: New Deals", level=2)
    insert_page_break_before(h)
    source_log.append(f"\n--- New Deals Series ---")
    
    # Page 9: Open Rate, Verified CTR
    add_chart_with_insights(doc, 'Deal Open Rate', CHART_CONFIG['Deal Open Rate'], source_log)
    add_chart_with_insights(doc, 'Deal Verified CTR', CHART_CONFIG['Deal Verified CTR'], source_log)
    
    # Page 10: Unsubscribe Rate
    add_chart_with_insights(doc, 'Deal Unsubscribe Rate', CHART_CONFIG['Deal Unsubscribe Rate'], source_log, page_break_before=True)
    
    # =========================================================================
    # OFF-MARKET (Pages 11-12)
    # =========================================================================
    h = doc.add_heading("Newsletter Series: Off-Market", level=2)
    insert_page_break_before(h)
    source_log.append(f"\n--- Off-Market Series ---")
    
    # Page 11: Open Rate, Verified CTR
    add_chart_with_insights(doc, 'Off-Market Open Rate', CHART_CONFIG['Off-Market Open Rate'], source_log)
    add_chart_with_insights(doc, 'Off-Market Verified CTR', CHART_CONFIG['Off-Market Verified CTR'], source_log)
    
    # Page 12: Unsubscribe Rate
    add_chart_with_insights(doc, 'Off-Market Unsubscribe Rate', CHART_CONFIG['Off-Market Unsubscribe Rate'], source_log, page_break_before=True)
    
    # =========================================================================
    # PODCASTS (Pages 13-14)
    # =========================================================================
    h = doc.add_heading("Newsletter Series: Podcasts", level=2)
    insert_page_break_before(h)
    source_log.append(f"\n--- Podcast Series ---")
    
    # Page 13: Open Rate, Verified CTR
    add_chart_with_insights(doc, 'Podcast Open Rate', CHART_CONFIG['Podcast Open Rate'], source_log)
    add_chart_with_insights(doc, 'Podcast Verified CTR', CHART_CONFIG['Podcast Verified CTR'], source_log)
    
    # Page 14: Unsubscribe Rate
    add_chart_with_insights(doc, 'Podcast Unsubscribe Rate', CHART_CONFIG['Podcast Unsubscribe Rate'], source_log, page_break_before=True)
    
    # =========================================================================
    # CASE STUDY (Pages 15-16)
    # =========================================================================
    h = doc.add_heading("Newsletter Series: Case Study", level=2)
    insert_page_break_before(h)
    source_log.append(f"\n--- Case Study Series ---")
    
    # Page 15: Open Rate, Verified CTR
    add_chart_with_insights(doc, 'Case Study Open Rate', CHART_CONFIG['Case Study Open Rate'], source_log)
    add_chart_with_insights(doc, 'Case Study Verified CTR', CHART_CONFIG['Case Study Verified CTR'], source_log)
    
    # Page 16: Unsubscribe Rate
    add_chart_with_insights(doc, 'Case Study Unsubscribe Rate', CHART_CONFIG['Case Study Unsubscribe Rate'], source_log, page_break_before=True)
    
    return doc


print("✓ Newsletter section function loaded")

✓ Newsletter section function loaded


### Section 14: Sales Section Generation

In [134]:
# =============================================================================
# SECTION 14: SALES SECTION GENERATION
# =============================================================================
# Creates the Sales Metrics section with Heading 1 title for TOC

def create_sales_section(doc, source_log):
    """Create Sales Metrics section with all charts and insights."""
    
    # Section 2 header - CENTERED, as Heading 1 (appears in TOC)
    add_section_title(doc, "Section 2: Sales Metrics", page_break_before=True)
    
    # =========================================================================
    # LEAD TIME & DEAL UPGRADE (Pages 17-18)
    # =========================================================================
    h = doc.add_heading("Lead Time & Deal Upgrade", level=2)
    insert_page_break_before(h)
    source_log.append(f"\n--- Lead Time & Deal Upgrade ---")
    
    # Page 17: Average Lead Time
    add_chart_with_insights(doc, 'Average Lead Time', CHART_CONFIG['Average Lead Time'], source_log)
    
    # Page 18: Deal Upgrade Visits, Deal Upgrade CVR
    add_chart_with_insights(doc, 'Deal Upgrade Visits', CHART_CONFIG['Deal Upgrade Visits'], source_log, page_break_before=True)
    add_chart_with_insights(doc, 'Deal Upgrade CVR', CHART_CONFIG['Deal Upgrade CVR'], source_log)
    
    # =========================================================================
    # PRO SITE FUNNEL (Page 19)
    # =========================================================================
    h = doc.add_heading("Pro Site Funnel", level=2)
    insert_page_break_before(h)
    source_log.append(f"\n--- Pro Site ---")
    
    # Page 19: Pro Site Visits, Pro Site CVR
    add_chart_with_insights(doc, 'Pro Site Visits', CHART_CONFIG['Pro Site Visits'], source_log)
    add_chart_with_insights(doc, 'Pro Site CVR', CHART_CONFIG['Pro Site CVR'], source_log)
    
    # =========================================================================
    # BOOKED CALLS (Page 20)
    # =========================================================================
    h = doc.add_heading("Booked Calls", level=2)
    insert_page_break_before(h)
    source_log.append(f"\n--- Booked Calls ---")
    
    # Page 20: Booked Calls - Closers, Booked Calls - Setters
    add_chart_with_insights(doc, 'Booked Calls - Closers (Discovery Call)', CHART_CONFIG['Booked Calls - Closers (Discovery Call)'], source_log)
    add_chart_with_insights(doc, 'Booked Calls - Setters (Intro Call)', CHART_CONFIG['Booked Calls - Setters (Intro Call)'], source_log)
    
    # =========================================================================
    # SALES PIPELINE: CALLS (Pages 21-22)
    # =========================================================================
    h = doc.add_heading("Sales Pipeline: Calls", level=2)
    insert_page_break_before(h)
    source_log.append(f"\n--- Sales Pipeline Calls ---")
    
    # Page 21: Scheduled Calls, Live Calls
    add_chart_with_insights(doc, 'Scheduled Calls', CHART_CONFIG['Scheduled Calls'], source_log)
    add_chart_with_insights(doc, 'Live Calls', CHART_CONFIG['Live Calls'], source_log)
    
    # Page 22: Show Rate
    add_chart_with_insights(doc, 'Show Rate', CHART_CONFIG['Show Rate'], source_log, page_break_before=True)
    
    # =========================================================================
    # OFFERS (Page 23)
    # =========================================================================
    h = doc.add_heading("Offers", level=2)
    insert_page_break_before(h)
    source_log.append(f"\n--- Offers ---")
    
    # Page 23: Offers, Offer Rate
    add_chart_with_insights(doc, 'Offers', CHART_CONFIG['Offers'], source_log)
    add_chart_with_insights(doc, 'Offer Rate', CHART_CONFIG['Offer Rate'], source_log)
    
    # =========================================================================
    # CLOSES (Page 24)
    # =========================================================================
    h = doc.add_heading("Closes", level=2)
    insert_page_break_before(h)
    source_log.append(f"\n--- Closes ---")
    
    # Page 24: Closes, Offer to Close Rate
    add_chart_with_insights(doc, 'Closes', CHART_CONFIG['Closes'], source_log)
    add_chart_with_insights(doc, 'Offer to Close Rate', CHART_CONFIG['Offer to Close Rate'], source_log)
    
    return doc


print("✓ Sales section function loaded")

✓ Sales section function loaded


### Section 15: Main Report Generation Function

In [135]:
# =============================================================================
# SECTION 15: MAIN REPORT GENERATION
# =============================================================================
# Orchestrates the complete MBR report generation process
# Sets the reporting month filter before processing any data

def generate_mbr_report():
    """
    Generate the complete Monthly Business Review report.
    
    Process:
    1. Load reference dates from analysis_ref_date.csv (Month Start/End)
    2. Set reporting month end date for data filtering
    3. Load monthly metrics tables for executive summary (if available)
    4. Create document with cover page
    5. Add executive summary
    6. Add newsletter section (charts + insights)
    7. Add sales section (charts + insights)
    8. Add page numbers
    9. Save DOCX to reports folder
    
    Output:
    - Monthly_Business_Review_<date_range>.docx
    - Prints source log showing all files used
    """
    print("=" * 60)
    print("GENERATING MONTHLY BUSINESS REVIEW (MBR) REPORT")
    print("=" * 60)
    
    source_log = []
    source_log.append("\n" + "=" * 60)
    source_log.append("DATA SOURCES USED IN THIS REPORT")
    source_log.append("=" * 60)
    
    # Step 1: Load reference dates (using Month Start/End for MBR)
    print("\n[Step 1] Loading reference dates...")
    date_info = parse_reference_dates(REF_DATE_FILE)
    if date_info is None:
        print("  ⚠ Could not load reference dates. Using default values.")
        date_info = {
            'month_start': 'Unknown',
            'month_end': 'Unknown',
            'date_range_filename': 'unknown_date',
            'date_range_display': 'Unknown Date Range'
        }
    print(f"  Reporting month: {date_info['date_range_display']}")
    source_log.append(f"\nReference Date File: analysis_ref_date.csv")
    source_log.append(f"  Reporting month: {date_info['date_range_display']}")
    
    # Step 2: Set reporting month end date for filtering
    print("\n[Step 2] Setting data filter...")
    set_reporting_month_end(date_info)
    
    # Step 3: Load metrics tables (if available, for executive summary)
    # Note: Using monthly metrics files if they exist
    print("\n[Step 3] Loading metrics tables...")
    newsletter_monthly_path = os.path.join(OUTPUTS_PATH, 'newsletter_analysis/monthly_newsletter_metrics.xlsx')
    sales_monthly_path = os.path.join(OUTPUTS_PATH, 'sales_metrics_analysis/monthly_sales_metrics.xlsx')
    
    newsletter_df = load_excel_metrics(newsletter_monthly_path)
    sales_df = load_excel_metrics(sales_monthly_path)
    
    print(f"  Newsletter metrics: {'✓ Loaded' if newsletter_df is not None else '⚠ Not found'}")
    print(f"  Sales metrics: {'✓ Loaded' if sales_df is not None else '⚠ Not found'}")
    
    # Step 4: Create document
    print("\n[Step 4] Creating document...")
    doc = Document()
    
    # Step 5: Cover page
    print("\n[Step 5] Adding cover page...")
    doc = create_cover_page(doc, date_info)
    
    # Step 6: Executive Summary
    print("\n[Step 6] Adding executive summary...")
    doc = create_executive_summary(doc, newsletter_df, sales_df)
    
    # Step 7: Newsletter section
    print("\n[Step 7] Adding newsletter section...")
    doc = create_newsletter_section(doc, source_log)
    
    # Step 8: Sales section
    print("\n[Step 8] Adding sales section...")
    doc = create_sales_section(doc, source_log)
    
    # Step 9: Add page numbers
    print("\n[Step 9] Adding page numbers...")
    add_page_numbers(doc)
    
    # Step 10: Save document
    print("\n[Step 10] Saving document...")
    filename = f"Monthly_Business_Review_{date_info['date_range_filename']}.docx"
    output_path = os.path.join(REPORTS_PATH, filename)
    doc.save(output_path)
    
    print(f"\n{'=' * 60}")
    print(f"✓ REPORT GENERATED SUCCESSFULLY!")
    print(f"{'=' * 60}")
    print(f"  File: {filename}")
    print(f"  Location: {output_path}")
    
    # Print source log showing all files used
    print("\n" + "\n".join(source_log))
    
    return output_path


print("✓ Main report generation function loaded")

✓ Main report generation function loaded


### Section 16: Execute Report Generation

In [136]:
# =============================================================================
# SECTION 16: EXECUTE REPORT GENERATION
# =============================================================================
# Run this cell to generate the Monthly Business Review report

# Generate the report
report_path = generate_mbr_report()

print(f"\n\n{'=' * 60}")
print("REPORT COMPLETE!")
print("=" * 60)
print(f"Open: {report_path}")

GENERATING MONTHLY BUSINESS REVIEW (MBR) REPORT

[Step 1] Loading reference dates...
  Reporting month: 11/01/2025 to 11/30/2025

[Step 2] Setting data filter...
  Data filter: Only including data up to 11/30/2025

[Step 3] Loading metrics tables...
  Newsletter metrics: ✓ Loaded
  Sales metrics: ✓ Loaded

[Step 4] Creating document...

[Step 5] Adding cover page...

[Step 6] Adding executive summary...

[Step 7] Adding newsletter section...

[Step 8] Adding sales section...

[Step 9] Adding page numbers...

[Step 10] Saving document...

✓ REPORT GENERATED SUCCESSFULLY!
  File: Monthly_Business_Review_11-01-2025_to_11-30-2025.docx
  Location: /Users/mariaangelicabaldres/Desktop/smb_dh_metrics/reports/Monthly_Business_Review_11-01-2025_to_11-30-2025.docx


DATA SOURCES USED IN THIS REPORT

Reference Date File: analysis_ref_date.csv
  Reporting month: 11/01/2025 to 11/30/2025

--- Landing Page Funnel Charts ---
  Chart: newsletter_monthly/monthly_visits.png
  Data: newsletter/monthly_new